# 03a — Seeded Multi-Run:  3 CNNs (5 seeds: 42/123/2025/7/99)

ConvNeXt-Tiny, Inception V3, Vanilla CNN, each trained 3x with fixed seeds for reproducibility + variance (mean±std). Every (model,seed) saves weights + preds immediately; re-running auto-skips completed runs. `num_workers=2` (CNNs are stable).

Attach: dataset + build_clean_split. Then Run All. Combine in 03c.

In [1]:
import os

SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"

# 1. Does the path exist?
print("SPLIT_DIR exists:", os.path.isdir(SPLIT_DIR))

# 2. Are the three index files + class names actually there?
if os.path.isdir(SPLIT_DIR):
    print("Contents:", os.listdir(SPLIT_DIR))
else:
    print("\nPath not found. Searching for the split files instead...")
    for r, d, f in os.walk('/kaggle/input'):
        if any(x.endswith('.npy') for x in f):
            print("  Found .npy files in:", r)

SPLIT_DIR exists: True
Contents: ['class_names.txt', '__results__.html', 'clean_train_indices.npy', 'clean_test_indices.npy', '__notebook__.ipynb', 'clean_val_indices.npy', '__output__.json', 'custom.css']


In [2]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
SEEDS     = [42, 123, 2025, 7, 99]
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5
NUM_WORKERS = 2

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f"Missing path: {pth}"
print("Paths OK | seeds:", SEEDS, "| workers:", NUM_WORKERS)

Paths OK | seeds: [42, 123, 2025, 7, 99] | workers: 2


In [3]:
import os
print(sorted(f for f in os.listdir('/kaggle/working') if f.endswith('.npz')))

[]


In [4]:
!pip install timm --quiet
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
    precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_idx = np.load(f"{SPLIT_DIR}/clean_train_indices.npy").tolist()
val_idx   = np.load(f"{SPLIT_DIR}/clean_val_indices.npy").tolist()
test_idx  = np.load(f"{SPLIT_DIR}/clean_test_indices.npy").tolist()
class_names = open(f"{SPLIT_DIR}/class_names.txt").read().splitlines()
SEVERE_IDX = class_names.index("Severe")
print("Classes:", class_names, "| Severe idx:", SEVERE_IDX)

Device: cuda
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'] | Severe idx: 4


In [5]:
# --- full determinism per run ---
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator(); g.manual_seed(seed)
    return g

In [6]:
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)), transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20), transforms.ColorJitter(0.3,0.3,0.2,0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, seed, crop_from=None):
    g = set_seed(seed)
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, generator=g),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NUM_WORKERS))

In [7]:
def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        run = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,"logits") else out, y)
            loss.backward(); optimizer.step()
            run += loss.item()
        if scheduler: scheduler.step()
        print(f"    epoch {ep+1}/{epochs} loss {run/len(loader):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        out = model(x.to(device))
        out = out.logits if hasattr(out,"logits") else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def save_and_report(name, seed, model, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    pr,rc,f1,sup = precision_recall_fscore_support(labels,preds,labels=range(NUM_CLASSES),zero_division=0)
    torch.save(model.state_dict(), f"{OUT_DIR}/{name}_seed{seed}.pth")
    np.savez(f"{OUT_DIR}/{name}_seed{seed}_preds.npz", probs=probs, preds=preds, labels=labels)
    print(f"  [{name} seed {seed}] acc {acc:.4f} | macroF1 {f1m:.4f} | "
          f"SevereRec {rc[SEVERE_IDX]:.3f} | ProlifRec {rc[class_names.index('Proliferate_DR')]:.3f}")
    print(f"  saved {name}_seed{seed}.pth + _preds.npz")

def already_done(name, seed):
    p = f"{OUT_DIR}/{name}_seed{seed}_preds.npz"
    if os.path.exists(p):
        print(f"  [skip] {name} seed {seed} already done"); return True
    return False

### ConvNeXt-Tiny × 5

In [8]:
for s in SEEDS:
    if already_done("convnext_tiny", s): continue
    print(f"ConvNeXt-Tiny seed {s}"); tr, te = make_loaders(224, s)
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES); m = m.to(device)
    opt = optim.Adam(m.parameters(), lr=1e-4)
    m = train_model(m, tr, nn.CrossEntropyLoss(), opt)
    probs, labels = evaluate(m, te); save_and_report("convnext_tiny", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

ConvNeXt-Tiny seed 42
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 172MB/s] 


    epoch 1/10 loss 0.2456
    epoch 2/10 loss 0.1001
    epoch 3/10 loss 0.0885
    epoch 4/10 loss 0.0751
    epoch 5/10 loss 0.0555
    epoch 6/10 loss 0.0369
    epoch 7/10 loss 0.0357
    epoch 8/10 loss 0.0588
    epoch 9/10 loss 0.0397
    epoch 10/10 loss 0.0296
  [convnext_tiny seed 42] acc 0.6716 | macroF1 0.6483 | SevereRec 0.494 | ProlifRec 0.213
  saved convnext_tiny_seed42.pth + _preds.npz
ConvNeXt-Tiny seed 123
    epoch 1/10 loss 0.2663
    epoch 2/10 loss 0.0974
    epoch 3/10 loss 0.0590
    epoch 4/10 loss 0.0593
    epoch 5/10 loss 0.0545
    epoch 6/10 loss 0.0569
    epoch 7/10 loss 0.0387
    epoch 8/10 loss 0.0455
    epoch 9/10 loss 0.0442
    epoch 10/10 loss 0.0376
  [convnext_tiny seed 123] acc 0.7183 | macroF1 0.6826 | SevereRec 0.310 | ProlifRec 0.823
  saved convnext_tiny_seed123.pth + _preds.npz
ConvNeXt-Tiny seed 2025
    epoch 1/10 loss 0.2872
    epoch 2/10 loss 0.0960
    epoch 3/10 loss 0.0684
    epoch 4/10 loss 0.0684
    epoch 5/10 loss 0.0529
  

### Inception V3 × 5

In [9]:
for s in SEEDS:
    if already_done("inception_v3", s): continue
    print(f"Inception V3 seed {s}"); tr, te = make_loaders(299, s, crop_from=320)
    m = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, NUM_CLASSES); m = m.to(device)
    opt = optim.Adam(m.parameters(), lr=1e-4)
    m = train_model(m, tr, nn.CrossEntropyLoss(), opt, aux=True)
    probs, labels = evaluate(m, te); save_and_report("inception_v3", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

Inception V3 seed 42
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 166MB/s] 


    epoch 1/10 loss 0.4335
    epoch 2/10 loss 0.1744
    epoch 3/10 loss 0.1515
    epoch 4/10 loss 0.1265
    epoch 5/10 loss 0.0944
    epoch 6/10 loss 0.0857
    epoch 7/10 loss 0.0804
    epoch 8/10 loss 0.0693
    epoch 9/10 loss 0.0510
    epoch 10/10 loss 0.0600
  [inception_v3 seed 42] acc 0.7743 | macroF1 0.7931 | SevereRec 0.874 | ProlifRec 0.354
  saved inception_v3_seed42.pth + _preds.npz
Inception V3 seed 123
    epoch 1/10 loss 0.4349
    epoch 2/10 loss 0.1764
    epoch 3/10 loss 0.1534
    epoch 4/10 loss 0.0996
    epoch 5/10 loss 0.0867
    epoch 6/10 loss 0.0731
    epoch 7/10 loss 0.0840
    epoch 8/10 loss 0.0618
    epoch 9/10 loss 0.0711
    epoch 10/10 loss 0.0539
  [inception_v3 seed 123] acc 0.9888 | macroF1 0.9854 | SevereRec 0.977 | ProlifRec 0.988
  saved inception_v3_seed123.pth + _preds.npz
Inception V3 seed 2025
    epoch 1/10 loss 0.4390
    epoch 2/10 loss 0.1665
    epoch 3/10 loss 0.1239
    epoch 4/10 loss 0.0977
    epoch 5/10 loss 0.1026
    epoc

In [10]:
import torch
print("cudnn.deterministic:", torch.backends.cudnn.deterministic)
print("cudnn.benchmark:", torch.backends.cudnn.benchmark)
print("cudnn.enabled:", torch.backends.cudnn.enabled)

cudnn.deterministic: True
cudnn.benchmark: False
cudnn.enabled: True


### Vanilla CNN × 5 (your original architecture, 224px)

In [11]:
class VanillaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,3,padding=1); self.conv2=nn.Conv2d(32,64,3,padding=1)
        self.conv3=nn.Conv2d(64,128,3,padding=1); self.conv4=nn.Conv2d(128,256,3,padding=1)
        self.relu=nn.ReLU(); self.maxPool=nn.MaxPool2d(2)
        self.dropout=nn.Dropout(0.25); self.flatten=nn.Flatten()
        self.fc1=nn.Linear(256*14*14,256); self.fc2=nn.Linear(256,num_classes)
    def forward(self,x):
        x=self.relu(self.maxPool(self.conv1(x))); x=self.relu(self.maxPool(self.conv2(x)))
        x=self.relu(self.maxPool(self.conv3(x))); x=self.relu(self.maxPool(self.conv4(x)))
        x=self.flatten(x); x=self.fc1(x); x=self.dropout(x); return self.fc2(x)

for s in SEEDS:
    if already_done("vanilla_cnn", s): continue
    print(f"Vanilla CNN seed {s}"); set_seed(s); tr, te = make_loaders(224, s)
    m = VanillaCNN().to(device); opt = optim.Adam(m.parameters(), lr=1e-3)
    m = train_model(m, tr, nn.CrossEntropyLoss(), opt)
    probs, labels = evaluate(m, te); save_and_report("vanilla_cnn", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

Vanilla CNN seed 42
    epoch 1/10 loss 0.9148
    epoch 2/10 loss 0.6734
    epoch 3/10 loss 0.5192
    epoch 4/10 loss 0.4202
    epoch 5/10 loss 0.3274
    epoch 6/10 loss 0.2834
    epoch 7/10 loss 0.2932
    epoch 8/10 loss 0.2581
    epoch 9/10 loss 0.2451
    epoch 10/10 loss 0.2349
  [vanilla_cnn seed 42] acc 0.7537 | macroF1 0.7330 | SevereRec 0.506 | ProlifRec 0.482
  saved vanilla_cnn_seed42.pth + _preds.npz
Vanilla CNN seed 123
    epoch 1/10 loss 0.9921
    epoch 2/10 loss 0.6775
    epoch 3/10 loss 0.5766
    epoch 4/10 loss 0.4721
    epoch 5/10 loss 0.3757
    epoch 6/10 loss 0.3338
    epoch 7/10 loss 0.2930
    epoch 8/10 loss 0.2752
    epoch 9/10 loss 0.2378
    epoch 10/10 loss 0.2490
  [vanilla_cnn seed 123] acc 0.7146 | macroF1 0.6571 | SevereRec 0.161 | ProlifRec 0.561
  saved vanilla_cnn_seed123.pth + _preds.npz
Vanilla CNN seed 2025
    epoch 1/10 loss 0.9352
    epoch 2/10 loss 0.6730
    epoch 3/10 loss 0.5541
    epoch 4/10 loss 0.4590
    epoch 5/10 loss 0

In [12]:
print("03a done. Save Version (Commit), then run 03b (Swin) and 03b2 (DeiT).")

03a done. Save Version (Commit), then run 03b (Swin) and 03b2 (DeiT).
